In [2]:
from data_utils import load_data, train_test_split_data

d = load_data()
X_train, X_test, Y_train, Y_test, Y_train_imp, Y_test_imp = train_test_split_data(
    d["X_svd"], d["Y_scaled"], d["Y_imputed"]
)
trait_cols = d["trait_cols"]

In [16]:
from pyfaidx import Fasta
import pysam

from rice_genome_query import FASTA_PATH
fasta_path = "../rice_data/GCA_rice.fasta"
# vcf_path = "../rice_data/RiceDiversity_44K_Genotypes_PLINK/sativas413_msu7.vcf"
vcf_path = "../rice_data/RiceDiversity_44K_Genotypes_PLINK/sativas413.vcf"

ref_fasta = Fasta(fasta_path)
vcf = pysam.VariantFile(vcf_path)
chrom_name = {i + 1: name for i, name in enumerate(ref_fasta.keys())}
print("Chromosomes in FASTA:", chrom_name)

Chromosomes in FASTA: {1: 'ENA|AP014957|AP014957.1', 2: 'ENA|AP014958|AP014958.1', 3: 'ENA|AP014959|AP014959.1', 4: 'ENA|AP014960|AP014960.1', 5: 'ENA|AP014961|AP014961.1', 6: 'ENA|AP014962|AP014962.1', 7: 'ENA|AP014963|AP014963.1', 8: 'ENA|AP014964|AP014964.1', 9: 'ENA|AP014965|AP014965.1', 10: 'ENA|AP014966|AP014966.1', 11: 'ENA|AP014967|AP014967.1', 12: 'ENA|AP014968|AP014968.1'}


In [47]:
def window_iterator(snp_pos_list, half_window=512, min_buffer=100):
    prev_start, prev_end = -float("inf"), -float("inf")
    for snp_pos in snp_pos_list:
        start = max(0, snp_pos - half_window)
        end   = snp_pos + half_window + 1
        if snp_pos > prev_start + min_buffer and snp_pos < prev_end - min_buffer:
            continue
        else:
            prev_start, prev_end = start, end
            yield start, end
    return start, end

In [48]:
# snp_pos_list = sorted([pos for pos, *_ in vcf.header.fetch(chrom_name[1])])
snp_pos = [(rec.chrom, rec.pos) for rec in vcf.fetch()]
from collections import defaultdict
snp_pos_dict = defaultdict(list)
for chrom, pos in snp_pos:
    snp_pos_dict[chrom].append(pos)
for chrom in snp_pos_dict:
    snp_pos_dict[chrom].sort()
window_dict = {}
HALF_WINDOW = 2048
for chrom, pos_list in snp_pos_dict.items():
    window_dict[chrom] = list(window_iterator(pos_list, half_window=HALF_WINDOW))
total_windows = sum(len(windows) for windows in window_dict.values())
print(f"Total windows: {total_windows}")

Total windows: 21254


In [ ]:
from collections import defaultdict
import pysam

HALF_WINDOW = 2048

snps_by_chrom = defaultdict(list)  # chrom → [(pos, alt_b, gt_alts_bytes)]
with pysam.VariantFile(vcf_path) as vcf:
    samples = list(vcf.header.samples)
    for rec in vcf.fetch():
        if not rec.alts:
            continue
        chrom_int = int(rec.chrom)
        gt_alts = bytes(
            1 if (rec.samples[s]["GT"] or (0,))[0] == 1 else 0
            for s in samples
        )
        snps_by_chrom[chrom_int].append((rec.pos, rec.alts[0][0], gt_alts))

# Must be sorted for the sliding window
for snp_list in snps_by_chrom.values():
    snp_list.sort(key=lambda x: x[0])

unique_windows = set()

for chrom_int, snp_list in snps_by_chrom.items():
    positions = [pos for pos, _, _ in snp_list]
    n = len(positions)

    for samp_idx in range(len(samples)):
        lo = hi = 0  # sliding window bounds (indices into snp_list)

        for i, (pos, alt_b, gt_alts) in enumerate(snp_list):
            # Advance sliding window
            while positions[lo] < pos - HALF_WINDOW:
                lo += 1
            while hi < n and positions[hi] < pos + HALF_WINDOW + 1:
                hi += 1

            # Alt SNP positions within this window for this sample
            alt_in_window = tuple(
                positions[j] for j in range(lo, hi)
                if snp_list[j][2][samp_idx]  # gt_alts[samp_idx]
            )
            unique_windows.add((chrom_int, pos, alt_in_window))

print(f"Unique windows: {len(unique_windows)}")


In [8]:
from collections import defaultdict
import pysam
from pyfaidx import Fasta
from scipy.sparse import csr_matrix

HALF_WINDOW = 512

ref_fasta = Fasta(fasta_path)
chrom_name = {i + 1: name for i, name in enumerate(ref_fasta.keys())}

# 1. Collect all SNPs per chromosome in one pass
# snps_by_chrom[chrom_int] = list of (pos, ref_byte, alt_byte, {sample_idx: gt})
snps_by_chrom = defaultdict(list)

with pysam.VariantFile(vcf_path) as vcf:
    samples = list(vcf.header.samples)
    for rec in vcf.fetch():
        if not rec.alts:
            continue
        chrom_int = int(rec.chrom)
        ref_b = rec.ref[0].encode()
        alt_b = rec.alts[0][0].encode()
        # store per-sample alt flags as a compact array
        gt_alts = bytes(
            1 if (rec.samples[s]["GT"] or (0,))[0] == 1 else 0
            for s in samples
        )
        snps_by_chrom[chrom_int].append((rec.pos, ref_b, alt_b, gt_alts))

# 2. For each chromosome, process one sample at a time
all_fragments = {}   # {sample_id: [fragment, ...]}

for chrom_int, snp_list in snps_by_chrom.items():
    seq_key  = chrom_name[chrom_int]
    ref_seq  = bytearray(str(ref_fasta[seq_key]).upper().encode())  # one copy

    for samp_idx, sample in enumerate(samples):
        # sample_seq = csr_matrix((1, len(ref_seq)), dtype="uint8")  # extremely cheap copy (~0 MB)
        sample_snps = []
        # Apply this sample's alt alleles
        for pos, ref_b, alt_b, gt_alts in snp_list:
            if gt_alts[samp_idx]:
                sample_snps.append((pos, alt_b[0]))

        # Extract windows around every SNP position
        window_left = max(0, sample_snps[0][0] - HALF_WINDOW)
        window_right = sample_snps[-1][0] + HALF_WINDOW + 1

        curr_slice_start = 0
        curr_slice_end = 0
        for pos, alt_b in sample_snps:
            if pos in range(window_left, window_right):
                curr_slice_end += 1
            else:
                if pos > window_right:
                    break
        
        frags = []
        for i, (pos, alt_b) in enumerate(sample_snps):
            window_left = max(0, pos - HALF_WINDOW)
            window_right   = pos + HALF_WINDOW + 1
            while sample_snps[curr_slice_start][0] < window_left:
                curr_slice_start += 1
            while curr_slice_end < len(sample_snps) and sample_snps[curr_slice_end][0] < window_right:
                curr_slice_end += 1
            frags.append((chrom_int, pos, alt_b, tuple(sample_snps[curr_slice_start:curr_slice_end])))

        all_fragments.setdefault(sample, []).extend(frags)

In [10]:
all_fragments_union = set()
for sample, frags in all_fragments.items():
    all_fragments_union.update(frags)
print(f"Total unique fragments: {len(all_fragments_union)}")

Total unique fragments: 3535178


Unique windows: 223738


In [3]:
from collections import defaultdict
import pysam
from pyfaidx import Fasta

HALF_WINDOW = 2048

ref_fasta = Fasta(fasta_path)
chrom_name = {i + 1: name for i, name in enumerate(ref_fasta.keys())}

# 1. Collect all SNPs per chromosome in one pass
# snps_by_chrom[chrom_int] = list of (pos, ref_byte, alt_byte, {sample_idx: gt})
snps_by_chrom = defaultdict(list)

with pysam.VariantFile(vcf_path) as vcf:
    samples = list(vcf.header.samples)
    for rec in vcf.fetch():
        if not rec.alts:
            continue
        chrom_int = int(rec.chrom)
        ref_b = rec.ref[0].encode()
        alt_b = rec.alts[0][0].encode()
        # store per-sample alt flags as a compact array
        gt_alts = bytes(
            1 if (rec.samples[s]["GT"] or (0,))[0] == 1 else 0
            for s in samples
        )
        snps_by_chrom[chrom_int].append((rec.pos, ref_b, alt_b, gt_alts))

# 2. For each chromosome, process one sample at a time
all_fragments = {}   # {sample_id: [fragment, ...]}

for chrom_int, snp_list in snps_by_chrom.items():
    seq_key  = chrom_name[chrom_int]
    ref_seq  = bytearray(str(ref_fasta[seq_key]).upper().encode())  # one copy

    for samp_idx, sample in enumerate(samples):
        sample_seq = bytearray(ref_seq)  # cheap copy (~40 MB)

        # Apply this sample's alt alleles
        for pos, ref_b, alt_b, gt_alts in snp_list:
            if gt_alts[samp_idx]:
                sample_seq[pos] = alt_b[0]

        # Extract windows around every SNP position
        frags = []
        for pos, *_ in snp_list:
            start = max(0, pos - HALF_WINDOW)
            end   = pos + HALF_WINDOW + 1
            window = sample_seq[start:end]
            if len(window) == 2 * HALF_WINDOW + 1:
                frags.append(bytes(window).decode())

        all_fragments.setdefault(sample, []).extend(frags)

        del sample_seq  # free before next sample

: 

In [ ]:
all_fragments_union = set()
for sample, frags in all_fragments.items():
    all_fragments_union.update(frags)
print(f"Total unique fragments: {len(all_fragments_union)}")

Total unique fragments: 90876


In [15]:
count = 0
with pysam.VariantFile(vcf_path) as vcf:
    samples = list(vcf.header.samples)  # 413 sample IDs
    print(f"Total samples in VCF: {len(samples)}")
    for record in vcf.fetch():
        chrom = record.chrom
        pos = record.pos
        ref = record.ref
        alt = record.alts[0] if record.alts else None
        count += len(record.samples)  # Count all sample entries for this variant
print(f"Total variant snps in VCF: {count}")

Total samples in VCF: 413
Total variant snps in VCF: 15240113


In [12]:
HALF_WINDOW = 16

with pysam.VariantFile(vcf_path) as vcf:
    samples = list(vcf.header.samples)  # 413 sample IDs
    
    for rec in vcf.fetch():
        chrom_int = int(rec.chrom)
        seq_key   = chrom_name[chrom_int]
        
        # Reference context window
        start  = max(0, rec.pos - HALF_WINDOW)
        window = ref_fasta[seq_key][start : rec.pos + HALF_WINDOW + 1].seq.upper()
        if len(window) != 2 * HALF_WINDOW + 1:
            continue
        
        ref_base = rec.ref
        alt_base = rec.alts[0] if rec.alts else None
        
        for sample in samples:
            gt = rec.samples[sample]["GT"]  # e.g. (0, 0) or (1, 1) or (None, None)
            
            if gt is None or None in gt:
                allele = ref_base          # treat missing as ref
            elif gt[0] == 1:
                allele = alt_base          # homozygous alt
            else:
                allele = ref_base          # homozygous ref
            
            # Substitute the center base with this sample's allele
            fragment = window[:HALF_WINDOW] + allele + window[HALF_WINDOW + 1:]
            
            # → use (sample, rec.id, fragment) here


In [3]:
HALF_WINDOW = 16

# Count num unique SNPs with context 
all_fragments = set()

from tqdm.notebook import tqdm

with pysam.VariantFile(vcf_path) as vcf:
    for rec in tqdm(vcf.fetch(), desc="Processing VCF records"):
        snp_id = rec.id

        chrom_int = int(rec.chrom) if rec.chrom.isdigit() else int(rec.chrom.lstrip("chr"))
        seq_key   = chrom_name[chrom_int]

        # rec.pos is 0-based MSU7 — slice directly, no offset needed
        start  = max(0, rec.pos - HALF_WINDOW)
        window = ref_fasta[seq_key][start : rec.pos + HALF_WINDOW + 1].seq.upper()

        if len(window) != 2 * HALF_WINDOW + 1:
            continue   # near chromosome boundary
        all_fragments.add(window)
print(f"Unique SNPs with context: {len(all_fragments)}")


Processing VCF records: 0it [00:00, ?it/s]

Unique SNPs with context: 35689
